In [ ]:
!pip install diffusers transformers accelerate controlnet_aux opencv-python-headless torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 8.3 MB/s eta 0:00:00


In [ ]:
import torch
from PIL import Image
from diffusers import (
    StableDiffusionXLControlNetPipeline,
    ControlNetModel,
    AutoencoderKL,
    UniPCMultistepScheduler
)
from diffusers.utils import load_image

# 1. Load an SDXL-compatible ControlNet for Scribble/Sketch
# Note: 'xinsir/controlnet-scribble-sdxl-1.0' is a high-quality SDXL scribble model
controlnet = ControlNetModel.from_pretrained(
    "xinsir/controlnet-scribble-sdxl-1.0",
    torch_dtype=torch.float16
)

# 2. Setup the SDXL Pipeline
# Using SDXL Base 1.0. You can also use SDXL-based realistic checkpoints.
model_id = "stabilityai/stable-diffusion-xl-base-1.0"
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    model_id,
    controlnet=controlnet,
    torch_dtype=torch.float16,
    use_safetensors=True
).to("cuda")

# 3. Load your SDXL-compatible LoRA
# Ensure this LoRA was trained for SDXL, not SD 1.5
pipe.load_lora_weights(
    "./",
    weight_name="UE5 interior render lora.safetensors"
)

# 4. Use an SDXL-optimized scheduler
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

# 5. Prepare your sketch
# SDXL is native to 1024x1024; using 512x512 may lower quality
sketch_image = load_image("f.png").resize((1024, 1024))

# 6. Generate
# SDXL often uses two prompts (prompt and prompt_2), but we can pass one to both
prompt = "modern minimalist room, cinematic lighting, oak wood textures, 8k uhd, architectural photography"
n_prompt = "low quality, blurry, messy, distorted, text, watermark, hand drawn, sketch"

image = pipe(
    prompt=prompt,
    negative_prompt=n_prompt,
    image=sketch_image,
    num_inference_steps=30,
    controlnet_conditioning_scale=0.8, # Recommended range for SDXL is 0.5 - 0.9
    cross_attention_kwargs={"scale": 0.8} # LoRA strength
).images[0]

image.save("interior_render_sdxl.png")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recomme

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/144 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/384 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/controlnet/pipeline_controlnet_sd_xl.py:928: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(
